**Import the necessary libraries to build a Transformer**

In [1]:
import logging
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow_datasets as tfds
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Dense, LayerNormalization, Dropout, MultiHeadAttention, Input
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split

***My original cell that I used to pull our training/test data I created as a .csv file. This data is 110 SQL prompts that I created, and then I utilized Claude to generate 8 unique Natural Language questions in different levels of basketball knowledge for each single prompt. This gives us 880 natural language prompts paired up with 110 different SQL queries. After we import this data we seperate it into a "questions" Dataframe and a "sql_full" Dataframe (Named as such because we add the necessary text to be able to immediately execute said query). 

From there we tokenize both dataframes, while making sure we maintain symbols that will be used so that we can have accurate SQL queries.

After we create our sequences, we then pad our sequences so that we can get the same size matrix so as not to conflict later. We also establish our vocab sizes for the model.

Finishing with a sanity check by printing each to confirm.***

In [ ]:
# df = pd.read_csv('sql_training_full.csv')

# questions = df["question"]
# sql_full = "<START> " + df["sql_query"] + " <END>"

# question_tokenizer = Tokenizer(oov_token="<UNK>")
# sql_tokenizer = Tokenizer(filters="", oov_token="<UNK>") # this keeps SQL symbols

# question_tokenizer.fit_on_texts(questions)
# sql_tokenizer.fit_on_texts(sql_full)

# question_sequences = question_tokenizer.texts_to_sequences(questions)
# sql_sequences = sql_tokenizer.texts_to_sequences(sql_full)

# encoder_inputs = pad_sequences(question_sequences, padding="post") 
# sql_sequences = pad_sequences(sql_sequences, padding="post")

# decoder_inputs = sql_sequences[:, :-1]
# targets = sql_sequences[:, 1:]

# input_vocab_size = len(question_tokenizer.word_index) + 1
# target_vocab_size = len(sql_tokenizer.word_index) + 1

# print("encoder_inputs:", encoder_inputs.shape)
# print("decoder_inputs:", decoder_inputs.shape)
# print("targets:", targets.shape)
# print("input_vocab_size:", input_vocab_size)
# print("target_vocab_size:", target_vocab_size)

encoder_inputs: (880, 16)
decoder_inputs: (880, 20)
targets: (880, 20)
input_vocab_size: 268
target_vocab_size: 114


***Fair Use AI: This section below I had Claude help with as we switched from using the .csv file I had originally provided to seperate test/train.txt files and I was not quite sure how to approach that with how the text files were formatted. The commented out section above is my original code for the .csv as the source.***

In [2]:
# ── Load from pre-split .txt files (tab-separated: question \t sql_query) ──
train_df = pd.read_csv('train.txt', sep='\t', header=None, names=['question', 'sql_query'])
test_df  = pd.read_csv('test.txt',  sep='\t', header=None, names=['question', 'sql_query'])

# Combine for tokenizer fitting (tokenizer needs to see the full vocab)
df = pd.concat([train_df, test_df], ignore_index=True)

questions = df["question"]
sql_full  = "<START> " + df["sql_query"] + " <END>"

question_tokenizer = Tokenizer(oov_token="<UNK>")
sql_tokenizer      = Tokenizer(filters="", oov_token="<UNK>")  # keeps SQL symbols

question_tokenizer.fit_on_texts(questions)
sql_tokenizer.fit_on_texts(sql_full)

# ── Encode each split separately ──
def encode_split(split_df):
    q_seqs  = question_tokenizer.texts_to_sequences(split_df["question"])
    sq_seqs = sql_tokenizer.texts_to_sequences("<START> " + split_df["sql_query"] + " <END>")
    enc_in  = pad_sequences(q_seqs,  padding="post")
    sq_seqs = pad_sequences(sq_seqs, padding="post")
    dec_in  = sq_seqs[:, :-1]
    tgts    = sq_seqs[:, 1:]
    return enc_in, dec_in, tgts

encoder_inputs,  decoder_inputs,  targets       = encode_split(train_df)
encoder_inputs_t, decoder_inputs_t, targets_t   = encode_split(test_df)

input_vocab_size  = len(question_tokenizer.word_index) + 1
target_vocab_size = len(sql_tokenizer.word_index) + 1

print("Train  — enc:", encoder_inputs.shape,   "dec:", decoder_inputs.shape,   "tgt:", targets.shape)
print("Test   — enc:", encoder_inputs_t.shape, "dec:", decoder_inputs_t.shape, "tgt:", targets_t.shape)
print("input_vocab_size:", input_vocab_size)
print("target_vocab_size:", target_vocab_size)

Train  — enc: (660, 16) dec: (660, 20) tgt: (660, 20)
Test   — enc: (220, 16) dec: (220, 20) tgt: (220, 20)
input_vocab_size: 268
target_vocab_size: 114


Below we create the model by building classes for the Encoder Layers, Encoder, Decoder Layers, Decoder, and then use these both together as a Transformer. 

First we create the positional encoding and positional embedding.

Positional Encoding - Creates a positional vector the same size as the word embedding so that we can determine the relative positon of each token in a sequence.

Positional Embedding - Creates a vector representation of the relationships of each token with different values so that words related to each other will be closer together and no relation far apart.

In [3]:
def positional_encoding(length, depth):
  depth = depth/2

  positions = np.arange(length)[:, np.newaxis]     # (seq, 1)
  depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth)

  angle_rates = 1 / (10000**depths)         # (1, depth)
  angle_rads = positions * angle_rates      # (pos, depth)

  pos_encoding = np.concatenate(
      [np.sin(angle_rads), np.cos(angle_rads)],
      axis=-1) 

  return tf.cast(pos_encoding, dtype=tf.float32)

class PositionalEmbedding(tf.keras.layers.Layer):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.d_model = d_model
    self.embedding = tf.keras.layers.Embedding(vocab_size, d_model, mask_zero=True) 
    self.pos_encoding = positional_encoding(length=2048, depth=d_model)

  def compute_mask(self, *args, **kwargs):
    return self.embedding.compute_mask(*args, **kwargs)

  def call(self, x):
    length = tf.shape(x)[1]
    x = self.embedding(x)
    # This factor sets the relative scale of the embedding and positonal_encoding.
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x = x + self.pos_encoding[tf.newaxis, :length, :]
    return x
  
  

Structure:\
-Encoder Layer - The actual layers needed to train the Encoder\
-Encoder - The encoder is designed to read the information we train it on and from there understand each representation of the data\
-Decoder Layer - The actual layers needed to train the Decoder (where our Cross-Attention is implemented)\
-Decoder - This acts as the mechanism that creates our SQL queries from the NL we input into the transformer.\
-Transformer - Each of these functions wrapped together so that they can train and pass information along (the Cross-Attention from Encoder to Decoder)

In [4]:
# Define Layer Params
#d_model = 64
#num_heads = 4
#key_dim = 32  Now calculated below in Classes
#ff_dim = 128
dropout = 0.01
#key_dim = d_model // num_heads

# Encoder Layer (MultiHead Attention > Add&Norm > Feed Forward > Add&Norm)
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, dropout, ff_dim, d_model):
        super(EncoderLayer, self).__init__()
        self.attention1 = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.dropout1 = Dropout(dropout)
        self.addnorm1 = LayerNormalization()

        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation='relu'),Dense(d_model)])

        self.dropout2 = Dropout(dropout)
        self.addnorm2 = LayerNormalization()

    def call(self, x, training=False):
        attn_output = self.attention1(query=x,key=x,value=x)
        attn_output = self.dropout1(attn_output, training=training)
        x = self.addnorm1(x + attn_output) # (x + attn_output) = the Add part of our Add & Norm
        
        ffn_output = self.ffn(x)
        ffn_output = self.dropout2(ffn_output, training=training)
        x = self.addnorm2(x + ffn_output) # (x + ffn_output) = the Add part of our Add & Norm

        return x


# Encoder (Run EncoderLayers, this is the reading/comprehension section of the Transformer model)
class Encoder(tf.keras.layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads, ff_dim, vocab_size, dropout):
        super().__init__()

        self.d_model = d_model
        self.num_layers = num_layers
        key_dim = d_model // num_heads

        self.pos_embedding = PositionalEmbedding(vocab_size=vocab_size, d_model=d_model)

        self.enc_layers = [EncoderLayer(num_heads=num_heads,key_dim=key_dim, ff_dim=ff_dim, d_model=d_model, dropout=dropout) for i in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(dropout)
    
    def call(self, x, training=False):
        x = self.pos_embedding(x)
        x = self.dropout(x, training=training)

        for layer in self.enc_layers:
            x = layer(x, training=training)

        return x


# Decoder Layer (Masked MultiHead Attention > Add&Norm > MultiHead Attention > Add&Norm)
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, dropout, ff_dim, d_model):
        super(DecoderLayer, self).__init__()
        self.masked_attention1 = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.dropout1 = Dropout(dropout)
        self.addnorm1 = LayerNormalization()

        self.cross_attention = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.dropout2 = Dropout(dropout)
        self.addnorm2 = LayerNormalization()

        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation='relu'),Dense(d_model)])

        self.dropout3 = Dropout(dropout)
        self.addnorm3 = LayerNormalization()
    
    def call(self, x, encoder_output, training=False):
        attn_output = self.masked_attention1(query=x, key=x, value=x, use_causal_mask=True)
        attn_output = self.dropout1(attn_output, training=training)
        x = self.addnorm1(x + attn_output)

        attn_output = self.cross_attention(query=x, key=encoder_output, value=encoder_output) # We add the Cross-Sectional here via the Encoder Output
        attn_output = self.dropout2(attn_output, training=training)
        x = self.addnorm2(x + attn_output)

        ffn_output = self.ffn(x)
        ffn_output = self.dropout3(ffn_output, training=training)
        x = self.addnorm3(x + ffn_output)

        return x

# Decoder (Run Decoder Layers while calling for Encoder Cross-Attention, this generates the Output for our Transformer model)
class Decoder(tf.keras.layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads, ff_dim, vocab_size, dropout=dropout):
        super(Decoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers
        key_dim = d_model // num_heads

        self.pos_embedding = PositionalEmbedding(vocab_size=vocab_size, d_model=d_model)
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.dec_layers = [DecoderLayer(num_heads=num_heads, key_dim=key_dim,ff_dim=ff_dim, d_model=d_model, dropout=dropout) for i in range(num_layers)]
    
    def call(self, x, encoder_output, training=False): # Call for Encoder Output for Cross-Attention
        x = self.pos_embedding(x)
        x = self.dropout(x, training=training)

        for i in range(self.num_layers):
            x = self.dec_layers[i](x, encoder_output, training=training)

        return x

# Transformer Model (Cross-Attention)
class Transformer(tf.keras.Model):
    def __init__(self, num_layers, num_heads, d_model, ff_dim, input_vocab_size, target_vocab_size, dropout):
        super().__init__()
        self.encoder = Encoder(num_layers=num_layers, d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, vocab_size=input_vocab_size, dropout=dropout)
        self.decoder = Decoder(num_layers=num_layers, d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, vocab_size=target_vocab_size, dropout=dropout)
        self.final_layer = tf.keras.layers.Dense(target_vocab_size)

    def call(self, inputs, training=False):
        context, x = inputs # Context in this case is our Natural Langauge Questions, and x is the SQL query we are trying to create.
        
        context = self.encoder(context, training=training)

        x = self.decoder(x, context, training=training)

        #Final Linear Layer Output:
        logits = self.final_layer(x)

        try:
            #Drop Keras Mask so it doesn't scale the Metrics and Losses
            del logits._keras_mask
        except AttributeError:
            pass

        return logits

Our Added Loss Functions:\
Masked Loss - Spares Catergorical Cross-Entropy on our MASKED data\
Masked Accuracy - Looks only at the accuracy of our MASKED data

In [5]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

def masked_loss(y_true, y_pred):
    loss = loss_object(y_true, y_pred)
    mask = tf.cast(y_true != 0, loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def masked_accuracy(y_true, y_pred):
    y_pred = tf.argmax(y_pred, axis=-1)
    y_pred = tf.cast(y_pred, y_true.dtype)

    match = tf.cast(y_true == y_pred, tf.float32)
    mask = tf.cast(y_true != 0, tf.float32)

    return tf.reduce_sum(match * mask) / tf.reduce_sum(mask)

Here we define our model parameters and compile the Transformer. I have ran this with d_model at 128 and 256. When ran at 128 we have a basline Key Dimension of 32 dimensions PER head of the Multi-Headed Attention, I raised the second run to 256 so that we have 64 Key Dims as that seems to historically worked well for other models such as BERT, GPT-2, and T5.

In [6]:
transformer = Transformer(num_layers=2, d_model=256, num_heads=4, ff_dim=512, input_vocab_size=input_vocab_size, target_vocab_size=target_vocab_size, dropout=0.1)

transformer.compile(optimizer="adam", loss=masked_loss, metrics=[masked_accuracy])

Add and Early Stop condition to prevent Overfitting. Then we chose the data we want to train and other parameters and begin training our model.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
# history = transformer.fit((encoder_inputs, decoder_inputs),targets, batch_size=32, epochs=20, validation_split=0.2)  **ORIGINAL MODEL WHEN USING .CSV DATASET**
history = transformer.fit([encoder_inputs, decoder_inputs], targets,validation_data=([encoder_inputs_t, decoder_inputs_t], targets_t), batch_size=32, epochs=20, validation_split=0.2)

Epoch 1/20


c:\Users\Sean\miniconda3\envs\ml\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'encoder_layer' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
c:\Users\Sean\miniconda3\envs\ml\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'decoder_layer' (of type DecoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


21/21 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - loss: 1.4672 - masked_accuracy: 0.6628 - val_loss: 0.5821 - val_masked_accuracy: 0.8115
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - loss: 0.4993 - masked_accuracy: 0.8428 - val_loss: 0.3983 - val_masked_accuracy: 0.8578
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - loss: 0.3910 - masked_accuracy: 0.8675 - val_loss: 0.3773 - val_masked_accuracy: 0.8631
Epoch 4/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - loss: 0.3283 - masked_accuracy: 0.8834 - val_loss: 0.2955 - val_masked_accuracy: 0.8966
Epoch 5/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - loss: 0.2267 - masked_accuracy: 0.9247 - val_loss: 0.1447 - val_masked_accuracy: 0.9532
Epoch 6/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - loss: 0.0927 - masked_accuracy: 0.9720 - val_loss: 0.0871 - val_masked_accuracy: 0.9696
Epoch 7/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - loss: 0.0458 - masked_accuracy: 0.9860 - val_loss: 0.0747 - val_masked_accuracy: 0.9768
Epoch 8/20
21/21 ━━━━━━

***Fair Use AI Notice: I used Claude to create the following predict_sql section so I could just have a quick visual output for my results initially. Once we being doing a full analysis I will create something on my own to do so.***

In [8]:
def predict_sql(question, max_len=50):
    # Encode the input question
    seq = question_tokenizer.texts_to_sequences([question])
    enc_in = pad_sequences(seq, maxlen=encoder_inputs.shape[1], padding="post")

    # Autoregressive decode
    start_token = sql_tokenizer.word_index["<start>"]
    end_token   = sql_tokenizer.word_index["<end>"]
    
    dec_input = np.array([[start_token]])
    result = []

    for _ in range(max_len):
        predictions = transformer.predict([enc_in, dec_input], verbose=0)
        next_token = np.argmax(predictions[:, -1, :], axis=-1)[0]
        
        if next_token == end_token:
            break
        result.append(next_token)
        dec_input = np.append(dec_input, [[next_token]], axis=1)

    # Convert token ids back to words
    idx2word = {v: k for k, v in sql_tokenizer.word_index.items()}
    return " ".join(idx2word.get(t, "<UNK>") for t in result)

# Test on a few examples from your test set
for i in range(10):
    q   = test_df["question"].iloc[i]
    gt  = test_df["sql_query"].iloc[i]
    pred = predict_sql(q)
    print(f"Q:    {q}")
    print(f"GT:   {gt}")
    print(f"PRED: {pred}")
    print()

with open("predictions.txt", "w") as f:
    for i in range(len(test_df)):
        q    = test_df["question"].iloc[i]
        gt   = test_df["sql_query"].iloc[i].strip()
        pred = predict_sql(q)
        match = "✓" if pred.strip().lower() == gt.lower() else "✗"
        
        f.write(f"{match} Q:    {q}\n")
        f.write(f"  GT:   {gt}\n")
        f.write(f"  PRED: {pred}\n")
        f.write("\n")
    
    # Summary at the bottom
    exact = sum(
        predict_sql(test_df["question"].iloc[i]).strip().lower() == test_df["sql_query"].iloc[i].strip().lower()
        for i in range(len(test_df))
    )
    f.write(f"Exact match: {exact}/{len(test_df)} = {exact/len(test_df):.2%}\n")

print("Saved to predictions.txt")

Q:    Who put up the biggest point total in one playoff game?
GT:   SELECT PLAYER_NAME FROM boxscores ORDER BY PTS DESC LIMIT 1;
PRED: select player_name from boxscores order by pts desc limit 1;

Q:    Name the player with the most points in any one playoff game.
GT:   SELECT PLAYER_NAME FROM boxscores ORDER BY PTS DESC LIMIT 1;
PRED: select player_name from boxscores order by pts desc limit 1;

Q:    Which Boston player came up shortest on the scoreboard in a playoff game?
GT:   SELECT PLAYER_NAME FROM boxscores WHERE TEAM_ABBREVIATION = 'BOS' ORDER BY PTS ASC LIMIT 1;
PRED: select player_name from boxscores where team_abbreviation = 'bos' order by pts desc limit 1;

Q:    Name the Boston player with the lowest point total in a playoff game.
GT:   SELECT PLAYER_NAME FROM boxscores WHERE TEAM_ABBREVIATION = 'BOS' ORDER BY PTS ASC LIMIT 1;
PRED: select player_name from boxscores where team_abbreviation = 'bos' order by pts asc limit 1;

Q:    Across the postseason, what's the average p